# K 近邻（KNN）

模式识别：几乎无训练——预测时找距离最近的 K 个训练点，投票决定类别。对应「离猫近就是猫」。

本笔记：小样本二维分类，手算距离与投票，对照 `sklearn.neighbors.KNeighborsClassifier`。**注意：特征尺度会影响距离。**


In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

# 二维小样本：类0 靠左下，类1 靠右上（间距拉开，避免距离并列）
X = np.array([
    [0.0, 0.0],
    [0.2, 0.1],
    [0.1, 0.3],
    [3.0, 3.0],
    [3.2, 2.8],
    [2.9, 3.1],
])
y = np.array([0, 0, 0, 1, 1, 1])
x_query = np.array([1.0, 1.0])
k = 3
print("训练点:\n", np.column_stack([X, y]))
print("查询点:", x_query, "K =", k)


## 1. 手算：欧氏距离 + 多数投票


In [ ]:
dists = np.linalg.norm(X - x_query, axis=1)
order = np.argsort(dists)
nn = order[:k]
print("距离:", np.round(dists, 4))
print("由近到远索引:", order)
print("最近 K 距离:", np.round(dists[nn], 4))
print("最近 K 个标签:", y[nn])
# 多数投票
votes, counts = np.unique(y[nn], return_counts=True)
y_manual = int(votes[np.argmax(counts)])
print("手算预测:", y_manual)


## 2. sklearn 对照


In [ ]:
clf = KNeighborsClassifier(n_neighbors=k, metric="euclidean")
clf.fit(X, y)
y_sk = clf.predict(x_query.reshape(1, -1))[0]
print("sklearn 预测:", y_sk)
print("一致?", y_manual == y_sk)


## 3. 尺度敏感（演示）

若把第二维放大 100 倍，距离几乎只由该维决定，预测可能改变——这是特征工程里「标准化」要解决的问题。


In [ ]:
# 查询点 [0.5, 1.6]：原始尺度近邻偏类0；把 y 放大 100 倍后近邻翻成类1
x_boundary = np.array([0.5, 1.6])
print("尺度演示查询点:", x_boundary)

dists_b = np.linalg.norm(X - x_boundary, axis=1)
nn_b = np.argsort(dists_b)[:k]
print("原始尺度 近邻标签:", y[nn_b], "多数类:", int(np.bincount(y[nn_b]).argmax()))

X_scaled = X.copy()
X_scaled[:, 1] *= 100
x_b2 = x_boundary.copy()
x_b2[1] *= 100
dists_s = np.linalg.norm(X_scaled - x_b2, axis=1)
nn_s = np.argsort(dists_s)[:k]
print("放大第2维后 近邻标签:", y[nn_s], "多数类:", int(np.bincount(y[nn_s]).argmax()))

clf_b = KNeighborsClassifier(n_neighbors=k).fit(X, y)
clf_s = KNeighborsClassifier(n_neighbors=k).fit(X_scaled, y)
print("sklearn 原始:", int(clf_b.predict(x_boundary.reshape(1, -1))[0]))
print("sklearn 放大y:", int(clf_s.predict(x_b2.reshape(1, -1))[0]))
print("→ 同一模型、同一点，仅特征尺度不同就可能改预测；实务上常先标准化。")
